**Table of contents**
1. [Set up and config](#1)
2. [Xử lý dữ liệu và EDA](#2)
3. [Mô hình và huấn luyện mô hình](#3)
4. [Đánh giá mô hình](#4)

# 1. Set up and config <a id='1'></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from abc import ABC, abstractmethod
from dataclasses import dataclass
import scipy
import sklearn
import random

In [ ]:
from utils import *

## Configuaration

In [3]:
# reproducibility
SEED = 42

# hyperparameters
LEARNING_RATE = .01
MAX_ITER = 10_000
REG = 1e-6
EPS = 1e-6
EPOCHS = 100
BATCH_SIZE = 100

# statistical significance level
ALPHA = .05

# data path
DATA_PATH = '../../data/raw/dry-bean-dataset/Dry_Bean_Dataset.xlsx'

In [4]:
def set_seed(seed: int) -> None:
  random.seed(seed)
  np.random.seed(seed)

set_seed(SEED)

# 2. Xử lý dữ liệu và EDA <a id='2'></a>
- Không validation:
$\text{Train} : \text{Test} = 80\% : 20\%$
- Có validation:
$\text{Train} : \text{Val} : \text{Test} = 60\% : 20\% : 20\%$

## 2.1 Xử lý dữ liệu

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. ĐỌC DỮ LIỆU TỪ FILE CSV ĐÃ LƯU
try:
    df = pd.read_csv('data/dataset.csv')
    print("Đã tải dữ liệu thành công.")
except FileNotFoundError:
    print("Lỗi: Không tìm thấy file 'dataset.csv'. Hãy chạy file xử lý trước đó.")
    exit()

# 2. TÁCH BIẾN ĐẶC TRƯNG (X) VÀ BIẾN MỤC TIÊU (y)
# Loại bỏ cột 'Class' (chuỗi) và 'Class_Encoded' (mục tiêu) ra khỏi X
X = df.drop(['Class', 'Class_Encoded'], axis=1)
y = df['Class_Encoded']

# 3. CHIA DỮ LIỆU: TRAIN (70%) / VAL (10%) / TEST (20%)
# Bước 1: Tách 20% cho Test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Bước 2: Tách 10% (trên tổng số) cho Validation từ phần 80% còn lại
# Tỷ lệ test_size = 0.125 vì 0.125 * 0.8 = 0.1 (tức 10%)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.125, random_state=42, stratify=y_train_val
)

# 4. CHUẨN HÓA DỮ LIỆU (STANDARDIZATION)
# Khởi tạo bộ chuẩn hóa
scaler = StandardScaler()

# LƯU Ý: Chỉ "fit" trên tập Train để tránh rò rỉ dữ liệu (Data Leakage)
X_train_scaled = scaler.fit_transform(X_train)

# "transform" cho tập Val và Test dựa trên thông số của tập Train
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 5. KIỂM TRA KẾT QUẢ
print("-" * 30)
print(f"Tổng số mẫu: {len(df)}")
print(f"Tập Huấn luyện (Train):    {X_train_scaled.shape}")
print(f"Tập Kiểm chứng (Val):     {X_val_scaled.shape}")
print(f"Tập Kiểm tra (Test):      {X_test_scaled.shape}")
print("-" * 30)

Đã tải dữ liệu thành công.
------------------------------
Tổng số mẫu: 13611
Tập Huấn luyện (Train):    (9527, 9)
Tập Kiểm chứng (Val):     (1361, 9)
Tập Kiểm tra (Test):      (2723, 9)
------------------------------


# Model

## Base

In [6]:
class Classification(ABC):
  @abstractmethod
  def fit(self, X: np.ndarray, y: np.ndarray) -> None:
    """
    Fit the model to the training data.
    """
    pass

  @abstractmethod
  def predict(self, X: np.ndarray) -> np.ndarray:
    """
    Predict the target values for the given input data.
    """
    pass

  @abstractmethod
  def evaluate(self, y_hat: np.ndarray, y: np.ndarray) -> float:
    """
    Evaluate the model on the given input and target data.
    """
    pass

## 3.1 Logistic Regression (nhị phân và đa lớp)

In [ ]:
class LogisticRegression(Classification):
  def __init__(
    self,
    random_state: int = SEED,
    solver: str = 'gradient_descent',
    learning_rate: float = LEARNING_RATE,
    max_iter: int = MAX_ITER,
  ):
    self.random_state = random_state
    self.solver = solver
    self.learning_rate = learning_rate
    self.max_iter = max_iter
    self.coef_ = None
    self.intercept_ = None
    self.history = {'loss': [], 'time': []}
    
  def fit(self, X: np.ndarray, y: np.ndarray) -> None:
    raise NotImplementedError('Not implemented yet')

  def predict(self, X: np.ndarray) -> np.ndarray:
    raise NotImplementedError('Not implemented yet')

  def evaluate(self, y_hat: np.ndarray, y: np.ndarray) -> float:
    raise NotImplementedError('Not implemented yet')

# Evaluation

# Logistic Regression (nhị phân và đa lớp):


Cài đặt Gradient Descent từ đầu cho nhị phân (sigmoid) và đa lớp (softmax).

Cài đặt Newton–Raphson / IRLS cho bài toán nhị phân và so sánh tốc độ
hội tụ (loss vs. epoch, loss vs. wall-clock time)

Cho bài toán đa lớp: so sánh ba chiến lược one-vs-rest, one-vs-one, và
softmax trực tiếp.

### Bảng so sánh các chiến lược phân lớp đa lớp

| Tiêu chí | One-vs-Rest (OvR) | One-vs-One (OvO) | Softmax Regression (Multinomial) |
| :--- | :--- | :--- | :--- |
| **Cơ chế hoạt động** | Huấn luyện $K$ bộ phân lớp nhị phân. Mỗi bộ phân biệt 1 lớp với tất cả các lớp còn lại. | Huấn luyện $K(K-1)/2$ bộ phân lớp. Mỗi bộ phân biệt giữa một cặp lớp $(i, j)$. | Huấn luyện **một** mô hình duy nhất với hàm kích hoạt Softmax để tính xác suất cho tất cả các lớp. |
| **Số lượng mô hình** | $K$ (Ví dụ: 3 lớp $\rightarrow$ 3 mô hình) | $K(K-1)/2$ (Ví dụ: 3 lớp $\rightarrow$ 3 mô hình; 10 lớp $\rightarrow$ 45 mô hình) | **1** (Mô hình hợp nhất) |
| **Độ phức tạp huấn luyện** | Thấp. Mỗi mô hình chạy trên toàn bộ tập dữ liệu. | Cao về số lượng mô hình, nhưng mỗi mô hình chỉ chạy trên dữ liệu của 2 lớp (nhanh hơn). | Trung bình. Cần tính toán ma trận trọng số lớn hơn và dùng hàm mất mát Cross-Entropy. |
| **Cách dự đoán** | Chọn lớp có xác suất (confidence score) cao nhất từ $K$ mô hình. | Bỏ phiếu (Voting). Lớp nào thắng nhiều cặp đấu nhất sẽ được chọn. | Chọn lớp có giá trị xác suất đầu ra lớn nhất (tổng các xác suất bằng 1). |
| **Ưu điểm** | Dễ cài đặt, trực quan, tốn ít tài nguyên bộ nhớ hơn OvO khi số lớp lớn. | Tránh được vấn đề mất cân bằng dữ liệu; hoạt động tốt với SVM hoặc các thuật toán nhạy cảm với kích thước dữ liệu. | Chính xác nhất về mặt toán học vì tối ưu hóa tất cả các lớp cùng lúc; không bị trùng lặp ranh giới. |
| **Nhược điểm** | Dễ bị mất cân bằng dữ liệu (lớp "Rest" thường chiếm đa số); các vùng ranh giới có thể bị chồng lấn. | Số lượng mô hình tăng rất nhanh khi số lớp tăng (bùng nổ tổ hợp). | Yêu cầu thay đổi cấu trúc hàm Loss và đạo hàm (phức tạp hơn khi tự code từ đầu). |

---

Trình bày đầy đủ đạo hàm Hessian, Jacobian của softmax.